# EXPERIMENT_7: Manifold-Steered Mask2Former
### Laparoscopic Liver Landmark Detection via Canonical Atlas & Visibility-Gated Steering

This notebook implements the complete **Manifold-Steered Mask2Former** architecture:
- **Tier 1:** 11-Query Canonical Atlas Head (Left/Right Lobes, Hinge, Boundaries)
- **Tier 2:** Visibility-Gated Cross-Attention Steering (VGS) with logarithmic attention silencing
- **Auxiliary:** Continuous 2D Riemannian Manifold Head (u, v regression with conditional loss masking)
- **Tier 3:** 9-Layer Mask2Former Transformer Decoder predicting dense multi-class landmark masks
- Full automatic evaluation on Validation (122 frames) and Test (109 frames) +  1-click download.

In [ ]:
!pip install -q transformers
import os, sys, time, glob, json, cv2, zipfile
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# NumPy 2.0 compatibility
for _a, _v in [("Inf", np.inf), ("NAN", np.nan), ("NaN", np.nan), ("PINF", np.inf), ("NINF", -np.inf)]:
    if not hasattr(np, _a): setattr(np, _a, _v)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def find_l3d_root():
    candidates = [
        "/kaggle/input/laparoscopic-liver-landmark-dataset/L3D",
        "/kaggle/input/laparoscopic-liver-landmarks/L3D",
        "/kaggle/input/l3d-dataset/L3D",
        "/kaggle/input/l3d/L3D",
        "data/L3D",
        "../data/L3D"
    ]
    for c in candidates:
        if os.path.exists(c):
            print(f"✅ Found L3D dataset at: {c}")
            return os.path.abspath(c)
    for p in glob.glob("/kaggle/input/**/labels", recursive=True):
        parent = os.path.dirname(p)
        if os.path.exists(os.path.join(parent, "Train")):
            print(f"✅ Found L3D dataset at: {parent}")
            return os.path.abspath(parent)
    return os.path.abspath("data/L3D")

L3D_ROOT = find_l3d_root()

In [ ]:
# Load code components directly from experiments/EXPERIMENT_7 or fallback inline
sys.path.insert(0, os.path.abspath("."))
from experiments.EXPERIMENT_7.utils.dataset import L3DManifoldDataset
from experiments.EXPERIMENT_7.models.manifold_steered_mask2former import ManifoldSteeredMask2Former
from experiments.EXPERIMENT_7.models.losses import ManifoldMultiTaskLoss
from experiments.EXPERIMENT_7.scripts.train import collate_fn_l3d, build_optimizer_and_scheduler
from experiments.EXPERIMENT_7.scripts.evaluate import run_evaluation, create_results_zip
print("✅ Successfully imported all EXPERIMENT_7 modules!")

In [ ]:
# Training Configuration (60 Epochs, Effective Batch Size: 4)
OUTPUT_DIR = "/kaggle/working/results_exp7"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS = 60
BATCH_SIZE = 2
ACCUM_STEPS = 2
LR_HEAD = 1e-4
LR_BACKBONE = 1e-5

train_dataset = L3DManifoldDataset(split="Train", data_dir=L3D_ROOT)
val_dataset = L3DManifoldDataset(split="Val", data_dir=L3D_ROOT, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn_l3d,
    pin_memory=True
)

model = ManifoldSteeredMask2Former().to(device)
criterion = ManifoldMultiTaskLoss(lambda_vis=1.0, lambda_coord=5.0, lambda_uv=1.0)
optimizer, scheduler = build_optimizer_and_scheduler(model, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD, epochs=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

In [ ]:
best_val_dice = -1.0
best_epoch = -1
best_model_path = os.path.join(OUTPUT_DIR, "best_model.pth")
training_log = []

print(f"🚀 Starting {EPOCHS}-Epoch Training...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    optimizer.zero_grad()
    running_loss, running_m2f, running_vis, running_coord, running_uv = 0.0, 0.0, 0.0, 0.0, 0.0
    num_b = 0
    
    for step, batch in enumerate(train_loader):
        imgs = batch["images"].to(device)
        mask_labels = [m.to(device) for m in batch["mask_labels"]]
        class_labels = [c.to(device) for c in batch["class_labels"]]
        gt_coords = batch["coords"].to(device)
        gt_vis = batch["visibilities"].to(device)
        gt_uv = batch["uv_maps"].to(device)
        gt_liver_mask = batch["liver_masks"].to(device)
        has_falcs = batch["has_falcs"].to(device)
        
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            outputs = model(pixel_values=imgs, mask_labels=mask_labels, class_labels=class_labels)
            loss, bd = criterion(
                m2f_loss=outputs["m2f_loss"],
                pred_coords=outputs["pred_coords"],
                pred_vis=outputs["pred_vis"],
                pred_uv=outputs["pred_uv"],
                gt_coords=gt_coords,
                gt_vis=gt_vis,
                gt_uv=gt_uv,
                gt_liver_mask=gt_liver_mask,
                has_falc=has_falcs
            )
            loss_scaled = loss / ACCUM_STEPS
            
        scaler.scale(loss_scaled).backward()
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        running_loss += bd["total_loss"]
        running_m2f += bd["m2f_loss"]
        running_vis += bd["loss_vis"]
        running_coord += bd["loss_coord"]
        running_uv += bd["loss_uv"]
        num_b += 1
        
    scheduler.step()
    ep_time = time.time() - t0
    avg_loss = running_loss / max(num_b, 1)
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({ep_time:.1f}s) | Loss: {avg_loss:.4f} (M2F: {running_m2f/num_b:.3f}, Vis: {running_vis/num_b:.3f}, Coord: {running_coord/num_b:.3f}, UV: {running_uv/num_b:.3f})")
    
    # Validation check
    val_summary, _ = run_evaluation(model, split="Val", data_dir=L3D_ROOT, out_dir=None, device=device, render_p40=False)
    val_dice = val_summary["macro_dice"]
    print(f"  👉 [Validation] Macro Dice: {val_dice*100:.2f}% | IoU: {val_summary['macro_iou']*100:.2f}% | ASSD: {val_summary['macro_assd']:.2f}px")
    
    training_log.append({"epoch": epoch, "train_loss": avg_loss, "val_macro_dice": val_dice, "val_macro_iou": val_summary["macro_iou"], "val_macro_assd": val_summary["macro_assd"]})
    pd.DataFrame(training_log).to_csv(os.path.join(OUTPUT_DIR, "training_log.csv"), index=False)
    
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_epoch = epoch
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "val_dice": val_dice, "val_summary": val_summary}, best_model_path)
        print(f"  ⭐ New Best Model saved! (Macro Dice: {best_val_dice*100:.2f}%)")

In [ ]:
# Final Full Benchmark & Zip Creation
print("
--- Running Full Final Evaluation on Best Checkpoint ---")
from experiments.EXPERIMENT_7.models.manifold_steered_mask2former import load_manifold_steered_model
best_model = load_manifold_steered_model(best_model_path, device=device)

val_sum, val_df = run_evaluation(best_model, split="Val", data_dir=L3D_ROOT, out_dir=OUTPUT_DIR, device=device, render_p40=True)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_predictions.csv"), index=False)

test_sum, test_df = run_evaluation(best_model, split="Test", data_dir=L3D_ROOT, out_dir=OUTPUT_DIR, device=device, render_p40=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_predictions.csv"), index=False)

summary_all = {"val_summary": val_sum, "test_summary": test_sum, "best_epoch": best_epoch}
with open(os.path.join(OUTPUT_DIR, "metrics_summary.json"), "w") as f:
    json.dump(summary_all, f, indent=4)

# Package results_exp7.zip
!cd /kaggle/working && zip -r results_exp7.zip results_exp7
print("🎉 All done! Download /kaggle/working/results_exp7.zip from the right sidebar!")